In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [7]:
class QuadState(TypedDict):
    a : int
    b : int
    c : int

    equation : str
    discriminant : float
    result : str


In [8]:
def show_equation(state: QuadState) :
    equation = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"

    return {'equation' : equation}


In [ ]:
def calculate_discriminant(state: QuadState):
    discriminant = state["b"]**2 - 4 * (state["a"] * state["c"])

    return {'discriminant' : discriminant}

In [10]:
def real_roots(state: QuadState):

    root1 = (-state["b"] + state["discriminant"]**0.5) / (2 * state["a"])
    root2 = (-state["b"] - state["discriminant"]**0.5) / (2 * state["a"])

    result = f"The roots of the equation are: {root1} and {root2}"

    return {'result' : result}

In [11]:
def no_real_roots(state: QuadState):

    result = "The equation has no real roots"
    return {"result": result}
    

In [12]:
def repeated_roots(state: QuadState):

    root = -state["b"] / (2 * state["a"])

    result = f"The equation has repeated real roots: {root}"
    return {"result": result}

In [19]:
def check_condition(state: QuadState):
    if state["discriminant"] > 0:
        return "real_roots"
    elif state["discriminant"] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

In [20]:
graph = StateGraph(QuadState)

graph.add_node('show_equation', show_equation)
graph.add_node('calclualte_discriminant', calclualte_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('no_real_roots', no_real_roots)
graph.add_node('repeated_roots', repeated_roots)

graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calclualte_discriminant')

graph.add_conditional_edges(
    'calclualte_discriminant',
    check_condition,
    {
        'real_roots': 'real_roots',
        'no_real_roots': 'no_real_roots',
        'repeated_roots': 'repeated_roots'
    }
)

graph.add_edge('real_roots', END)
graph.add_edge('no_real_roots', END)
graph.add_edge('repeated_roots', END)

workflow = graph.compile()

In [21]:
initial_state = {
    "a": 1,
    "b": -3,
    "c": 2
}

workflow.invoke(initial_state)

{'a': 1,
 'b': -3,
 'c': 2,
 'equation': '1x^2 + -3x + 2 = 0',
 'discriminant': 1,
 'result': 'The roots of the equation are: 2.0 and 1.0'}